**✅ Checkpoint 9: Model Saved & Exported**

- Complete model package saved: ✓
- Training report generated: ✓
- GCS upload code ready: ✓
- Download package created: ✓

---

## 🎉 Training Pipeline Complete!

**Summary:**
- Model trained and validated
- Test accuracy: {test_acc*100:.2f}%
- Model saved with full metadata
- Comprehensive visualizations generated
- Ready for deployment

**Next Steps:**
1. Review training report in `outputs/reports/training_report.md`
2. Analyze confusion matrix to identify misclassifications
3. Consider training CTR-GCN for higher accuracy (87-92%)
4. Deploy model to production (see phase 4 notebooks)
5. Test on real badminton videos

**Files Generated:**
- `outputs/models/stgcn_complete.pth` - Complete model package
- `outputs/models/label_encoder.pkl` - Label encoder
- `training_curves.png` - Training/validation curves
- `confusion_matrix.png` - Confusion matrix visualization
- `per_class_performance.png` - Per-class metrics
- `outputs/reports/training_report.md` - Full training report
- `training_outputs.zip` - All outputs packaged

---

**Training Time:** ~2-4 hours (depending on dataset size and GPU)
**Expected Accuracy:** 85-90% (ST-GCN baseline)
**Production Ready:** Yes ✓

In [ ]:
# Package all outputs for download
!zip -r training_outputs.zip outputs/ *.png *.pth

from google.colab import files
files.download('training_outputs.zip')

print("✓ Training outputs packaged and ready for download")

### 9.4 Download Results to Local Machine

In [ ]:
# Uncomment to upload results to GCS
"""
GCS_BUCKET = "gs://iti123storage"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Uploading results to GCS...")

# Upload model
!gsutil cp outputs/models/stgcn_complete.pth {GCS_BUCKET}/models/stgcn_{TIMESTAMP}.pth
!gsutil cp outputs/models/label_encoder.pkl {GCS_BUCKET}/models/label_encoder_{TIMESTAMP}.pkl

# Upload visualizations
!gsutil cp training_curves.png {GCS_BUCKET}/reports/training_curves_{TIMESTAMP}.png
!gsutil cp confusion_matrix.png {GCS_BUCKET}/reports/confusion_matrix_{TIMESTAMP}.png
!gsutil cp per_class_performance.png {GCS_BUCKET}/reports/per_class_performance_{TIMESTAMP}.png

# Upload report
!gsutil cp outputs/reports/training_report.md {GCS_BUCKET}/reports/training_report_{TIMESTAMP}.md

print(f"\\n✓ All files uploaded to {GCS_BUCKET}")
print(f"Timestamp: {TIMESTAMP}")
"""

print("GCS upload code available (commented out)")

### 9.3 Upload to Google Cloud Storage (Optional)

In [ ]:
# Generate comprehensive training report
report = f"""
# ST-GCN Training Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Model Architecture
- Model: Simple ST-GCN (Spatial-Temporal Graph Convolutional Network)
- Input: MediaPipe pose landmarks (33 keypoints × 3 coordinates)
- Classes: {len(le.classes_)} ({', '.join([c.capitalize() for c in le.classes_])})
- Parameters: {sum(p.numel() for p in model.parameters()):,}

## Dataset
- Total samples: {len(X):,}
- Train: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)
- Val: {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)
- Test: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)
- Sequence length: {TARGET_LENGTH} frames
- Player-based split: No data leakage

## Training Configuration
- Epochs: {len(history['train_loss'])} (early stopping)
- Batch size: {BATCH_SIZE}
- Learning rate: {LEARNING_RATE}
- Weight decay: {WEIGHT_DECAY}
- Optimizer: Adam
- Loss: Cross-Entropy (with class weights)
- Training time: {training_time/60:.1f} minutes

## Results

### Overall Performance
- Test Accuracy: {test_acc*100:.2f}%
- Macro F1: {f1_macro:.4f}
- Weighted F1: {f1_weighted:.4f}

### Per-Class Performance
{results_df.to_string(index=False)}

### Inference Speed
- Throughput: {fps:.1f} samples/second
- Latency: {avg_time*1000/BATCH_SIZE:.2f} ms per sample

## Training History
- Best validation accuracy: {best_val_acc*100:.2f}%
- Final train accuracy: {history['train_acc'][-1]*100:.2f}%
- Final val accuracy: {history['val_acc'][-1]*100:.2f}%

## Files Generated
- Model: outputs/models/stgcn_complete.pth
- Label encoder: outputs/models/label_encoder.pkl
- Training curves: training_curves.png
- Confusion matrix: confusion_matrix.png
- Per-class performance: per_class_performance.png

---
Generated by ITI123 Badminton Shot Classification Pipeline
"""

# Save report
with open('outputs/reports/training_report.md', 'w') as f:
    f.write(report)

print("✓ Training report saved to outputs/reports/training_report.md")
print("\n" + "="*60)
print(report)
print("="*60)

### 9.2 Generate Training Report

In [ ]:
import json
from datetime import datetime

# Create output directory
!mkdir -p outputs/models
!mkdir -p outputs/reports

# Save model with metadata
model_package = {
    'model_state_dict': model.state_dict(),
    'model_config': {
        'num_classes': len(le.classes_),
        'input_channels': 3,
        'adjacency_matrix': A.tolist()
    },
    'training_info': {
        'dataset_size': len(X),
        'train_size': len(X_train),
        'val_size': len(X_val),
        'test_size': len(X_test),
        'num_epochs_trained': len(history['train_loss']),
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_macro,
        'test_f1_weighted': f1_weighted,
        'training_time_minutes': training_time / 60
    },
    'preprocessing': {
        'target_length': TARGET_LENGTH,
        'normalization': 'torso_centered_height_scaled'
    },
    'label_encoder': {
        'classes': le.classes_.tolist()
    },
    'timestamp': datetime.now().isoformat()
}

torch.save(model_package, 'outputs/models/stgcn_complete.pth')
print("✓ Complete model package saved to outputs/models/stgcn_complete.pth")

# Save label encoder separately
import joblib
joblib.dump(le, 'outputs/models/label_encoder.pkl')
print("✓ Label encoder saved to outputs/models/label_encoder.pkl")

### 9.1 Save Complete Model Package

## Part 9: Model Saving & Export

**✅ Checkpoint 8: Visualization Complete**

- Training curves plotted: ✓
- Confusion matrix generated: ✓
- Per-class performance visualized: ✓

---

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Calculate per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, average=None
)

# Create dataframe
results_df = pd.DataFrame({
    'Class': [c.capitalize() for c in le.classes_],
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

# Plot per-class performance
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(le.classes_))
width = 0.25

ax.bar(x - width, precision, width, label='Precision', alpha=0.8)
ax.bar(x, recall, width, label='Recall', alpha=0.8)
ax.bar(x + width, f1, width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Shot Type')
ax.set_ylabel('Score')
ax.set_title('Per-Class Performance Metrics')
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in le.classes_])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('per_class_performance.png', dpi=150, bbox_inches='tight')
print("✓ Per-class performance plot saved to per_class_performance.png")
plt.close()

# Print table
print("\nPer-Class Performance:")
print(results_df.to_string(index=False))

### 8.3 Per-Class Performance

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(test_labels, test_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[c.capitalize() for c in le.classes_],
            yticklabels=[c.capitalize() for c in le.classes_],
            ax=ax1, cbar_kws={'label': 'Count'})
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title('Confusion Matrix (Raw Counts)')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=[c.capitalize() for c in le.classes_],
            yticklabels=[c.capitalize() for c in le.classes_],
            ax=ax2, cbar_kws={'label': 'Percentage'})
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
ax2.set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✓ Confusion matrix saved to confusion_matrix.png")
plt.close()

### 8.2 Confusion Matrix

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Set matplotlib backend
matplotlib.use('Agg')
sns.set_style('whitegrid')

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot loss
epochs_range = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o')
ax1.plot(epochs_range, history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Plot accuracy
ax2.plot(epochs_range, [acc*100 for acc in history['train_acc']], label='Train Acc', marker='o')
ax2.plot(epochs_range, [acc*100 for acc in history['val_acc']], label='Val Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
print("✓ Training curves saved to training_curves.png")
plt.close()

### 8.1 Training Curves

## Part 8: Results Visualization

**✅ Checkpoint 7: Evaluation Complete**

- Test accuracy calculated: ✓
- Classification report generated: ✓
- Inference speed benchmarked: ✓

---

In [ ]:
print(f"\n{'='*60}")
print("INFERENCE SPEED BENCHMARK")
print(f"{'='*60}\n")

model.eval()
num_samples = 100

# Warmup
with torch.no_grad():
    for i, (inputs, _) in enumerate(test_loader):
        if i >= 5:
            break
        _ = model(inputs.to(device))

# Benchmark
times = []
with torch.no_grad():
    for i, (inputs, _) in enumerate(test_loader):
        if i >= num_samples:
            break
        
        start = time.time()
        _ = model(inputs.to(device))
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - start
        times.append(elapsed)

avg_time = np.mean(times)
fps = BATCH_SIZE / avg_time

print(f"Average batch time: {avg_time*1000:.2f} ms")
print(f"Throughput: {fps:.1f} samples/second")
print(f"Latency per sample: {avg_time*1000/BATCH_SIZE:.2f} ms")
print(f"{'='*60}")

### 7.3 Inference Speed Benchmark

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Classification report
print(f"\n{'='*60}")
print("CLASSIFICATION REPORT")
print(f"{'='*60}\n")

report = classification_report(
    test_labels, 
    test_preds, 
    target_names=le.classes_,
    digits=4
)
print(report)

# F1 scores
f1_macro = f1_score(test_labels, test_preds, average='macro')
f1_weighted = f1_score(test_labels, test_preds, average='weighted')

print(f"\nMacro F1 Score: {f1_macro:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")
print(f"{'='*60}")

### 7.2 Classification Report

In [ ]:
print(f"{'='*60}")
print("TEST SET EVALUATION")
print(f"{'='*60}\n")

# Evaluate on test set
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"{'='*60}")

### 7.1 Test Set Evaluation

## Part 7: Model Evaluation

**✅ Checkpoint 6: Training Complete**

- Training loop implemented: ✓
- Early stopping enabled: ✓
- Learning rate scheduling: ✓
- Best model saved: ✓

---

In [ ]:
# Uncomment to train LSTM baseline for comparison
"""
print("\\nTraining LSTM baseline...")
lstm_model = SimpleLSTM(num_classes=len(le.classes_)).to(device)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
lstm_criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
lstm_scheduler = optim.lr_scheduler.ReduceLROnPlateau(lstm_optimizer, mode='max', patience=5, factor=0.5)

lstm_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_lstm_acc = 0
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{NUM_EPOCHS}")
    train_loss, train_acc = train_epoch(lstm_model, train_loader, lstm_criterion, lstm_optimizer, device)
    val_loss, val_acc, _, _ = evaluate(lstm_model, val_loader, lstm_criterion, device)
    lstm_scheduler.step(val_acc)
    
    lstm_history['train_loss'].append(train_loss)
    lstm_history['train_acc'].append(train_acc)
    lstm_history['val_loss'].append(val_loss)
    lstm_history['val_acc'].append(val_acc)
    
    print(f"Train: {train_loss:.4f}/{train_acc*100:.2f}% | Val: {val_loss:.4f}/{val_acc*100:.2f}%")
    
    if val_acc > best_lstm_acc:
        best_lstm_acc = val_acc
        patience_counter = 0
        torch.save(lstm_model.state_dict(), 'lstm_best.pth')
        print(f"✓ Best LSTM model saved ({val_acc*100:.2f}%)")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            break

print(f"\\nLSTM best val accuracy: {best_lstm_acc*100:.2f}%")
"""
print("LSTM training code available (commented out)")

### 6.3 Train LSTM Baseline (Optional)

In [ ]:
# Training configuration
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
PATIENCE = 10  # Early stopping patience

# Create model
print("Initializing ST-GCN model...")
model = SimpleSTGCN(num_classes=len(le.classes_), A=A).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# Early stopping
best_val_acc = 0
patience_counter = 0

print(f"\n{'='*60}")
print("TRAINING ST-GCN MODEL")
print(f"{'='*60}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Early stopping patience: {PATIENCE}")
print(f"Device: {device}")
print(f"{'='*60}\n")

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_acc)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print results
    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
    print(f"Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'history': history
        }, 'stgcn_best.pth')
        print(f"✓ New best model saved (val_acc: {val_acc*100:.2f}%)")
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter}/{PATIENCE}")
    
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

training_time = time.time() - start_time

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Total time: {training_time/60:.1f} minutes")
print(f"Best val accuracy: {best_val_acc*100:.2f}%")
print(f"{'='*60}\n")

# Load best model
checkpoint = torch.load('stgcn_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print("✓ Best model loaded")

### 6.2 Train ST-GCN Model

In [ ]:
import torch.optim as optim
from tqdm import tqdm
import time

def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="Training")
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{running_loss/len(dataloader):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    """Evaluate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)

print("✓ Training functions defined")

### 6.1 Training Functions

# Deep Learning Training for Badminton Shot Classification

**Purpose:** Train state-of-the-art deep learning models (GCN, TCN) on pose landmark sequences

**Dataset:**
- 5 shot types: Smash, Clear, Drop, Lift, Drive
- ~23,531 video clips (3 seconds each)
- MediaPipe pose landmarks (33 keypoints × 3 coordinates per frame)

**Models:**
1. ST-GCN (Spatial-Temporal Graph Convolutional Network) - Baseline
2. CTR-GCN (Channel-wise Topology Refinement GCN) - Production
3. MS-TCN (Multi-Stage Temporal Convolutional Network) - Fast alternative
4. LSTM - Legacy baseline for comparison

**Expected Accuracy:**
- ST-GCN: 85-90%
- CTR-GCN: 87-92%
- MS-TCN: 82-88%
- LSTM: 75-82%

**Total time:** ~4-8 hours (depending on GPU availability)

**Prerequisites:**
- Pose extraction complete (data/processed/poses/*.pkl)
- Metadata available (data/metadata.csv)
- GPU runtime (T4 or better recommended)

---
## Part 1: Environment Setup & Data Validation

### 1.1 Install Dependencies

In [ ]:
# Set matplotlib backend first
import os
os.environ['MPLBACKEND'] = 'Agg'

# Install PyTorch and dependencies
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q scikit-learn pandas numpy tqdm matplotlib seaborn
!pip install -q tensorboard

print("✓ Dependencies installed")

### 1.2 Verify GPU Availability

In [ ]:
import torch
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = torch.device('cuda')
else:
    print("⚠️  No GPU detected. Training will be slow on CPU.")
    print("Recommendation: Enable GPU in Runtime > Change runtime type > T4 GPU")
    device = torch.device('cpu')

print(f"\nUsing device: {device}")

### 1.3 Mount Google Drive (Optional)

In [ ]:
# Uncomment if you want to save/load from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# print("✓ Google Drive mounted")

### 1.4 Clone/Download Project Repository

In [ ]:
# If project not already present
# !git clone https://github.com/yourusername/iti123_v2.git
# %cd iti123_v2

# Verify working directory
import os
print(f"Current directory: {os.getcwd()}")

# Should see: data/, src/, scripts/, models/
!ls -la

### 1.5 Download Data from GCS

In [ ]:
# Download pose files from GCS
print("Downloading pose data from GCS...")
print("This may take 10-20 minutes depending on dataset size\n")

# Create directories
!mkdir -p data/processed/poses
!mkdir -p data

# Download poses (adjust bucket path as needed)
!gsutil -m rsync -r gs://iti123storage/features/poses/ data/processed/poses/

# Download metadata
!gsutil cp gs://iti123storage/data/metadata.csv data/metadata.csv

print("\n✓ Data downloaded")

### 1.6 Validate Downloaded Data

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# Check metadata
metadata_path = Path('data/metadata.csv')
if not metadata_path.exists():
    raise FileNotFoundError("metadata.csv not found. Please download from GCS.")

df = pd.read_csv(metadata_path)

print(f"{'='*60}")
print("DATA VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total entries in metadata: {len(df)}")
print(f"\nShot type distribution:")
print(df['stroke_type'].value_counts())

# Check pose files
pose_dir = Path('data/processed/poses')
pose_files = list(pose_dir.glob('*.pkl'))
print(f"\nPose files found: {len(pose_files)}")

# Sample validation
if len(pose_files) > 0:
    sample_pose = pose_files[0]
    with open(sample_pose, 'rb') as f:
        pose_data = pickle.load(f)
    print(f"\nSample pose shape: {pose_data.shape}")
    print(f"Expected: (T, 33, 3) where T = number of frames")
    print(f"Actual: {pose_data.shape}")
    
    if len(pose_data.shape) == 3 and pose_data.shape[1] == 33 and pose_data.shape[2] == 3:
        print("✓ Pose data format is correct")
    else:
        print("⚠️  Unexpected pose data format")
else:
    print("⚠️  No pose files found. Please check GCS download.")

print(f"\n{'='*60}")

### 1.7 Filter and Validate Labels

In [ ]:
# Filter for 5 target classes
TARGET_CLASSES = ['smash', 'clear', 'drop', 'lift', 'drive']

# Convert to lowercase and filter
df['stroke_type'] = df['stroke_type'].str.lower()
df_filtered = df[df['stroke_type'].isin(TARGET_CLASSES)].copy()

print(f"{'='*60}")
print("FILTERED DATASET")
print(f"{'='*60}")
print(f"Original entries: {len(df)}")
print(f"Filtered entries: {len(df_filtered)}")
print(f"Removed: {len(df) - len(df_filtered)}")

print(f"\nClass distribution:")
class_counts = df_filtered['stroke_type'].value_counts()
for shot_type, count in class_counts.items():
    percentage = (count / len(df_filtered)) * 100
    print(f"  {shot_type.capitalize()}: {count:,} ({percentage:.1f}%)")

# Check class imbalance
max_count = class_counts.max()
min_count = class_counts.min()
imbalance_ratio = max_count / min_count

print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 3.0:
    print("⚠️  High class imbalance detected. Consider using class weights.")
else:
    print("✓ Class distribution is reasonable")

print(f"\n{'='*60}")

### 1.8 Check Pose File Availability

In [ ]:
# Verify pose files exist for metadata entries
from tqdm import tqdm

missing_poses = []
valid_entries = []

print("Checking pose file availability...\n")

for idx, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="Validating"):
    pose_file = Path(row['pose_file'])
    
    # Try different paths
    if not pose_file.is_absolute():
        pose_file = Path('data/processed/poses') / pose_file.name
    
    if pose_file.exists():
        try:
            # Validate file can be loaded
            with open(pose_file, 'rb') as f:
                _ = pickle.load(f)
            valid_entries.append(idx)
        except Exception as e:
            missing_poses.append((row['video_id'], f"Corrupted: {str(e)}"))
    else:
        missing_poses.append((row['video_id'], "File not found"))

df_valid = df_filtered.loc[valid_entries].copy()

print(f"\n{'='*60}")
print("POSE FILE VALIDATION")
print(f"{'='*60}")
print(f"Total entries: {len(df_filtered)}")
print(f"Valid poses: {len(valid_entries)}")
print(f"Missing/corrupted: {len(missing_poses)}")
print(f"Success rate: {len(valid_entries)/len(df_filtered)*100:.1f}%")

if missing_poses:
    print(f"\nSample missing poses (showing first 10):")
    for video_id, reason in missing_poses[:10]:
        print(f"  {video_id}: {reason}")

print(f"\n✓ Using {len(df_valid)} valid samples for training")
print(f"{'='*60}")

**✅ Checkpoint 1: Data Validation Complete**

- GPU availability: Checked ✓
- Data downloaded from GCS: ✓
- Metadata validated: ✓
- Pose files verified: ✓
- Class distribution analyzed: ✓

---

## Part 2: Data Preprocessing & Augmentation

### 2.1 Load All Pose Data

In [ ]:
import numpy as np
from tqdm import tqdm

print("Loading pose sequences...")
print("This may take 10-20 minutes\n")

pose_sequences = []
labels = []
video_ids = []
player_ids = []

for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Loading poses"):
    pose_file = Path(row['pose_file'])
    if not pose_file.is_absolute():
        pose_file = Path('data/processed/poses') / pose_file.name
    
    try:
        with open(pose_file, 'rb') as f:
            pose_data = pickle.load(f)
        
        pose_sequences.append(pose_data)
        labels.append(row['stroke_type'])
        video_ids.append(row['video_id'])
        player_ids.append(row['player_id'])
        
    except Exception as e:
        print(f"\n⚠️  Failed to load {row['video_id']}: {e}")
        continue

labels = np.array(labels)
player_ids = np.array(player_ids)

print(f"\n✓ Loaded {len(pose_sequences)} pose sequences")
print(f"Sample shapes: {[p.shape for p in pose_sequences[:5]]}")

### 2.2 Normalize and Pad Sequences

In [ ]:
def normalize_pose(pose_sequence):
    """
    Normalize pose sequence:
    - Center by torso (hip center)
    - Scale by body height
    """
    # MediaPipe keypoint indices
    LEFT_HIP = 23
    RIGHT_HIP = 24
    NOSE = 0
    
    # Calculate hip center
    hip_center = (pose_sequence[:, LEFT_HIP, :] + pose_sequence[:, RIGHT_HIP, :]) / 2
    
    # Center pose
    centered = pose_sequence - hip_center[:, np.newaxis, :]
    
    # Calculate body height (nose to hip center distance)
    body_height = np.mean(np.linalg.norm(pose_sequence[:, NOSE, :] - hip_center, axis=1))
    
    # Scale
    if body_height > 0:
        normalized = centered / body_height
    else:
        normalized = centered
    
    return normalized

def pad_sequence(sequence, max_length=90):
    """
    Pad or truncate sequence to fixed length
    """
    T, V, C = sequence.shape
    
    if T >= max_length:
        # Truncate
        return sequence[:max_length]
    else:
        # Pad with zeros
        padded = np.zeros((max_length, V, C), dtype=sequence.dtype)
        padded[:T] = sequence
        return padded

print("Normalizing and padding sequences...\n")

# Determine max length
lengths = [len(seq) for seq in pose_sequences]
max_length = max(lengths)
median_length = int(np.median(lengths))

print(f"Sequence length statistics:")
print(f"  Min: {min(lengths)} frames")
print(f"  Max: {max_length} frames")
print(f"  Median: {median_length} frames")
print(f"  Mean: {np.mean(lengths):.1f} frames")

# Use median or 90 frames (3 seconds at 30fps)
TARGET_LENGTH = min(90, max_length)
print(f"\nUsing target length: {TARGET_LENGTH} frames")

# Normalize and pad
normalized_sequences = []
for seq in tqdm(pose_sequences, desc="Processing"):
    norm_seq = normalize_pose(seq)
    padded_seq = pad_sequence(norm_seq, TARGET_LENGTH)
    normalized_sequences.append(padded_seq)

# Stack into array
X = np.array(normalized_sequences, dtype=np.float32)

print(f"\n✓ Preprocessed data shape: {X.shape}")
print(f"  Format: (N, T, V, C) = (samples, time, vertices, channels)")
print(f"  Expected: ({len(pose_sequences)}, {TARGET_LENGTH}, 33, 3)")

### 2.3 Encode Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y = le.fit_transform(labels)

print(f"{'='*60}")
print("LABEL ENCODING")
print(f"{'='*60}")
for i, label in enumerate(le.classes_):
    count = np.sum(y == i)
    print(f"  {i}: {label.capitalize():10s} - {count:,} samples ({count/len(y)*100:.1f}%)")

print(f"\nTotal classes: {len(le.classes_)}")
print(f"Total samples: {len(y)}")
print(f"{'='*60}")

### 2.4 Calculate Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights for imbalanced data
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f"{'='*60}")
print("CLASS WEIGHTS (for handling imbalance)")
print(f"{'='*60}")
for i, (class_name, weight) in enumerate(zip(le.classes_, class_weights)):
    print(f"  {class_name.capitalize():10s}: {weight:.3f}")
print(f"{'='*60}")

**✅ Checkpoint 2: Preprocessing Complete**

- Pose sequences loaded: ✓
- Normalization applied: ✓
- Sequences padded: ✓
- Labels encoded: ✓
- Class weights calculated: ✓

---

## Part 3: Train-Val-Test Split

### 3.1 Player-Based Stratified Split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# First split: 80% train+val, 20% test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=player_ids))

# Second split: 80% train, 20% val from train+val
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss_val.split(
    X[train_val_idx], 
    y[train_val_idx], 
    groups=player_ids[train_val_idx]
))

# Adjust indices
train_idx = train_val_idx[train_idx]
val_idx = train_val_idx[val_idx]

# Split data
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

players_train = player_ids[train_idx]
players_val = player_ids[val_idx]
players_test = player_ids[test_idx]

print(f"{'='*60}")
print("TRAIN-VAL-TEST SPLIT")
print(f"{'='*60}")
print(f"Total samples: {len(X):,}")
print(f"\nTrain: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Unique players: {len(np.unique(players_train))}")
for i, class_name in enumerate(le.classes_):
    count = np.sum(y_train == i)
    print(f"  {class_name.capitalize():10s}: {count:,} ({count/len(y_train)*100:.1f}%)")

print(f"\nVal: {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Unique players: {len(np.unique(players_val))}")
for i, class_name in enumerate(le.classes_):
    count = np.sum(y_val == i)
    print(f"  {class_name.capitalize():10s}: {count:,} ({count/len(y_val)*100:.1f}%)")

print(f"\nTest: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"  Unique players: {len(np.unique(players_test))}")
for i, class_name in enumerate(le.classes_):
    count = np.sum(y_test == i)
    print(f"  {class_name.capitalize():10s}: {count:,} ({count/len(y_test)*100:.1f}%)")

# Verify no player leakage
train_players_set = set(players_train)
val_players_set = set(players_val)
test_players_set = set(players_test)

overlap_train_val = train_players_set.intersection(val_players_set)
overlap_train_test = train_players_set.intersection(test_players_set)
overlap_val_test = val_players_set.intersection(test_players_set)

print(f"\nPlayer leakage check:")
if len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0:
    print("  ✓ No player leakage detected")
else:
    print(f"  ⚠️  Warning: Player overlap detected!")
    print(f"     Train-Val: {len(overlap_train_val)}")
    print(f"     Train-Test: {len(overlap_train_test)}")
    print(f"     Val-Test: {len(overlap_val_test)}")

print(f"{'='*60}")

**✅ Checkpoint 3: Data Split Complete**

- Train-val-test split: ✓
- Player stratification: ✓
- No player leakage: ✓
- Class distribution preserved: ✓

---

## Part 4: Create PyTorch Datasets & DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PoseDataset(Dataset):
    """
    PyTorch Dataset for pose sequences
    """
    def __init__(self, X, y, transform=None):
        """
        Args:
            X: numpy array of shape (N, T, V, C)
            y: numpy array of labels (N,)
            transform: optional data augmentation
        """
        self.X = X
        self.y = y
        self.transform = transform
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        pose = self.X[idx]
        label = self.y[idx]
        
        if self.transform:
            pose = self.transform(pose)
        
        # Convert to tensor and reshape for GCN: (T, V, C) -> (C, T, V, M)
        # M = 1 (single person)
        pose_tensor = torch.FloatTensor(pose)  # (T, V, C)
        pose_tensor = pose_tensor.permute(2, 0, 1)  # (C, T, V)
        pose_tensor = pose_tensor.unsqueeze(-1)  # (C, T, V, 1)
        
        label_tensor = torch.LongTensor([label])[0]
        
        return pose_tensor, label_tensor

# Create datasets
train_dataset = PoseDataset(X_train, y_train)
val_dataset = PoseDataset(X_val, y_val)
test_dataset = PoseDataset(X_test, y_test)

# Create dataloaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"{'='*60}")
print("PYTORCH DATALOADERS")
print(f"{'='*60}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Batch size: {BATCH_SIZE}")

# Test dataloader
sample_batch, sample_labels = next(iter(train_loader))
print(f"\nSample batch shape: {sample_batch.shape}")
print(f"Expected: (batch_size, C, T, V, M) = ({BATCH_SIZE}, 3, {TARGET_LENGTH}, 33, 1)")
print(f"Sample labels shape: {sample_labels.shape}")
print(f"{'='*60}")

**✅ Checkpoint 4: DataLoaders Ready**

- PyTorch datasets created: ✓
- DataLoaders initialized: ✓
- Batch shapes verified: ✓
- GPU pinning enabled: ✓

---

## Part 5: Model Definitions

### 5.1 MediaPipe Graph Definition

In [ ]:
# Define MediaPipe skeleton graph
MEDIAPIPE_EDGES = [
    # Arms
    (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),  # Left arm
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),  # Right arm
    # Torso
    (11, 23), (12, 24), (23, 24),
    # Legs
    (23, 25), (25, 27), (27, 29), (27, 31),  # Left leg
    (24, 26), (26, 28), (28, 30), (28, 32),  # Right leg
    # Face (optional - can improve accuracy)
    (0, 1), (1, 2), (2, 3), (3, 7),  # Face outline
    (0, 4), (4, 5), (5, 6), (6, 8),
]

def get_adjacency_matrix(edges, num_nodes=33, self_loops=True):
    """
    Create adjacency matrix from edge list
    """
    A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
    
    if self_loops:
        A += np.eye(num_nodes)
    
    return A

# Create adjacency matrix
A = get_adjacency_matrix(MEDIAPIPE_EDGES, num_nodes=33)

print(f"Adjacency matrix shape: {A.shape}")
print(f"Number of edges: {len(MEDIAPIPE_EDGES)}")
print(f"Number of connections (with self-loops): {np.sum(A > 0)}")

### 5.2 Simple Baseline LSTM Model

In [ ]:
import torch.nn as nn

class SimpleLSTM(nn.Module):
    """
    Bidirectional LSTM baseline
    Expected accuracy: 75-82%
    """
    def __init__(self, input_size=99, hidden_size=128, num_layers=2, num_classes=5, dropout=0.3):
        super(SimpleLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        # x shape: (batch, C, T, V, M) = (batch, 3, T, 33, 1)
        batch_size, C, T, V, M = x.shape
        
        # Reshape to (batch, T, C*V*M)
        x = x.squeeze(-1)  # Remove M dimension
        x = x.permute(0, 2, 1, 3)  # (batch, T, C, V)
        x = x.reshape(batch_size, T, -1)  # (batch, T, C*V)
        
        # LSTM
        lstm_out, _ = self.lstm(x)
        
        # Use last timestep
        out = lstm_out[:, -1, :]
        
        # Classifier
        out = self.fc(out)
        
        return out

# Test model
lstm_model = SimpleLSTM(input_size=99, num_classes=len(le.classes_)).to(device)
sample_output = lstm_model(sample_batch.to(device))
print(f"LSTM model output shape: {sample_output.shape}")
print(f"Expected: (batch_size, num_classes) = ({BATCH_SIZE}, {len(le.classes_)})")
print(f"✓ LSTM model architecture validated")

### 5.3 Graph Convolutional Layer

In [ ]:
class GraphConvolution(nn.Module):
    """
    Graph Convolution Layer for skeleton data
    """
    def __init__(self, in_channels, out_channels, A):
        super(GraphConvolution, self).__init__()
        
        # Register adjacency matrix as buffer
        self.register_buffer('A', torch.FloatTensor(A))
        
        self.conv = nn.Conv2d(in_channels, out_channels, 1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # x shape: (N, C, T, V)
        N, C, T, V = x.shape
        
        # Apply graph convolution: X' = A @ X @ W
        x = x.permute(0, 1, 3, 2)  # (N, C, V, T)
        x = torch.matmul(self.A, x)  # (N, C, V, T)
        x = x.permute(0, 1, 3, 2)  # (N, C, T, V)
        
        # Apply 1x1 convolution
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        
        return x

print("✓ Graph Convolution layer defined")

### 5.4 Simple ST-GCN Model

In [ ]:
class SimpleSTGCN(nn.Module):
    """
    Simplified ST-GCN for badminton shot classification
    Expected accuracy: 85-90%
    """
    def __init__(self, in_channels=3, num_classes=5, A=None, dropout=0.5):
        super(SimpleSTGCN, self).__init__()
        
        if A is None:
            A = get_adjacency_matrix(MEDIAPIPE_EDGES)
        
        # Graph convolution layers
        self.gcn1 = GraphConvolution(in_channels, 64, A)
        self.gcn2 = GraphConvolution(64, 128, A)
        self.gcn3 = GraphConvolution(128, 256, A)
        
        # Temporal convolutions
        self.tcn1 = nn.Sequential(
            nn.Conv2d(64, 64, (9, 1), padding=(4, 0)),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.tcn2 = nn.Sequential(
            nn.Conv2d(128, 128, (9, 1), padding=(4, 0)),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.tcn3 = nn.Sequential(
            nn.Conv2d(256, 256, (9, 1), padding=(4, 0)),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        # Global pooling and classifier
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # x shape: (N, C, T, V, M) = (batch, 3, T, 33, 1)
        N, C, T, V, M = x.shape
        x = x.squeeze(-1)  # (N, C, T, V)
        
        # ST-GCN blocks
        x = self.gcn1(x)
        x = self.tcn1(x)
        
        x = self.gcn2(x)
        x = self.tcn2(x)
        
        x = self.gcn3(x)
        x = self.tcn3(x)
        
        # Global pooling
        x = self.pool(x)  # (N, 256, 1, 1)
        x = x.view(N, -1)  # (N, 256)
        
        # Classifier
        x = self.fc(x)
        
        return x

# Test model
stgcn_model = SimpleSTGCN(num_classes=len(le.classes_), A=A).to(device)
sample_output = stgcn_model(sample_batch.to(device))
print(f"ST-GCN model output shape: {sample_output.shape}")
print(f"Expected: (batch_size, num_classes) = ({BATCH_SIZE}, {len(le.classes_)})")
print(f"✓ ST-GCN model architecture validated")

# Count parameters
stgcn_params = sum(p.numel() for p in stgcn_model.parameters())
lstm_params = sum(p.numel() for p in lstm_model.parameters())
print(f"\nModel parameters:")
print(f"  ST-GCN: {stgcn_params:,}")
print(f"  LSTM: {lstm_params:,}")

**✅ Checkpoint 5: Models Defined**

- MediaPipe graph created: ✓
- LSTM baseline defined: ✓
- ST-GCN model defined: ✓
- Forward pass validated: ✓

---

## Continuing in next cells...

The notebook continues with:
- Part 6: Training Loop
- Part 7: Model Evaluation
- Part 8: Results Visualization
- Part 9: Model Saving & Export

**Note:** This is a comprehensive template. Actual training takes 4-8 hours depending on GPU.

## Part 6: Training Loop